In [ ]:
using LinearAlgebra
using LinearOperators
using SparseArrays

In [ ]:
function get_1d_laplace_op_matrix(n)
	off = ones(n-1)
	diag = ones(n)
	spdiagm(-1 => off, 0 => -2diag, 1 => off)
end

In [ ]:
# L
# E L, L E
# E E L, E L E, L E E
# E E E L, E E L E, E L E E, L E E E
# (E E E L), (E E L) (E), (E L) (E E), (L) (E E E)
# E ... E L = L of size n^(k+1) if E is there k times
function get_laplace_op_matrix(n, d)
	L_full = get_1d_laplace_op_matrix(n^d)
	for k in 1:(d-1)
		L = get_1d_laplace_op_matrix(n^(d-k))
		size = n^k
		E = sparse(I, size, size)
		L_full += kron(L, E)
	end
	L_full
end

# 4 points, 3 dims
#get_laplace_op_matrix(4, 3)

In [ ]:
#Matrix(L_full)

## define RHS and solve using different methods

$\Delta u = f \:\:\text{on}\:\:\Omega$

$u = g \:\:\text{on}\:\:\partial\Omega$

supp 1D:

$ \partial u / \partial x \approx \frac{u(x+h/2) - u(x-h/2)}{h}$

$\Delta u(x) \approx \frac{u(x+h) - 2u(x) + u(x-h)}{h^2}$

$ u_{i-1} - 2u_i + u_{i+1} = h^2 f_i$

$ i = 0,1,...,n,n+1 $

boundary: $\:i=0, \:i=n+1 $

we know $\:u_0, \:u_{n+1}\:$ from BC

we need $n$ equation for $n$ unknowns ($u_1,...,u_n$)

$ u_0 - 2u_1 + u_2 = h^2 f_1 $

$ u_1 - 2u_2 + u_3 = h^2 f_2 $

...

$ u_{n-2} - 2u_{n-1} + u_{n} = h^2 f_{n-1} $

$ u_{n-1} - 2u_n + u_{n+1} = h^2 f_n $

we get

$ L U = F $

where
- $L$ matrix of size $n \times n$
- $U = (u_1, u_2, ..., u_{n-1}, u_n)$
- $F = h^2(f_1, f_2, ..., f_{n-1}, f_n) - (u_0, 0, ..., 0, u_{n+1})$

In [ ]:
n = 100
d = 4
N = n^d

In [ ]:
h = 1 / (n+1)

In [ ]:
function u_analytic_fun(x)
    #prod(x)*prod(1 .- x)
    prod(sin.(pi*x))
end

function f_fun(x)
    d = length(x)
    -d*pi^2 * prod(sin.(pi*x))
end

### fiddling around

In [ ]:
v = Vector([1,2,3])
v, f_fun(v)

In [ ]:
v = Vector([Vector([1,2,3]), Vector([1,1,1])])
v, f_fun.(v)

In [ ]:
xs = [range(h, 1-h; length=n) for _ in 1:d]

In [ ]:
coords = collect(Iterators.product(xs...))

In [ ]:
collect(coords[1])

In [ ]:
[collect(coords[1]), collect(coords[2])]

In [ ]:
f_fun.([collect(coords[1]), collect(coords[2])])

In [ ]:
#grid_points_as_1d_vect = [collect(x) for x in coords[1:end]]
#f_fun.(grid_points_as_1d_vect)

### here we go

In [ ]:
function get_grid_points_as_1d_vect(n, d)
    a = 0
    b = 1
    h = 1/(n+1)
    xs = [range(h, 1-h; length=n) for _ in 1:d]
    coords = collect(Iterators.product(xs...))
    [collect(x) for x in coords[1:end]]
end
#f.(grid_points_as_1d_vect)

In [ ]:
n^d

In [ ]:
sdf

In [ ]:
grid_points_as_1d_vect = get_grid_points_as_1d_vect(n,d);

In [ ]:
U_analytic = u_analytic_fun.(grid_points_as_1d_vect);

In [ ]:
f = f_fun.(grid_points_as_1d_vect);
F = h^2 * f;

In [ ]:
L = get_laplace_op_matrix(n,d);

In [ ]:
using IterativeSolvers
U_cg = cg(L, F)

In [ ]:
U_direct = L \ F;

In [ ]:
using CairoMakie
CairoMakie.activate!()

In [ ]:
#grid_points = collect(range(h,1-h, length=n));
#scatter(grid_points, U_analytic, color=:gray0)
#lines!(grid_points, U_direct, color=:royalblue1)
#lines!(grid_points, U_cg, color=:gray0)
#current_figure()

In [ ]:
#n_trials = 8
#n_mult = 8
#max_jacobi = zeros(n_trials)
#max_gs = zeros(n_trials)
#max_direct = zeros(n_trials)
#
#for i in 1:n_trials
#    n = n_mult*i
#    h = 1/(n+1)
#    grid_points = collect(range(h,1-h, length=n))
#    L = get_laplace_op_matrix(n)
#    F = h^2 * f_fun(grid_points)
#    U_jacobi = jacobi(L, F)
#    U_gs = gauss_seidel(L, F; maxiter=100)
#    U_direct = L \ F
#
#    max_jacobi[i] = maximum(U_jacobi)
#    max_gs[i] = maximum(U_gs)
#    max_direct[i] = maximum(U_direct)
#end